# Section 4: High NA Circular Aperture (ε = 0)

**System:** NA = 0.9, λ = 532 nm, circular aperture (no obscuration), medium: air (n = 1).

At high NA, the paraxial approximation breaks down and **vectorial diffraction theory** is essential. The Richards-Wolf formalism correctly accounts for:
- Depolarization: the x-polarized input generates a cross-polarized Ey component at the focus
- Longitudinal field: a significant Ez component appears along the optical axis
- Tighter focus: the focal spot is smaller than the paraxial (Airy) prediction
- Asymmetric focal spot: x-polarized beams give an elliptical spot (elongated in x)

**Goals:**
- Focal-plane 1D radial intensity for uniform and Gaussian (α=1,2,4) inputs
- Show Ex, Ey, Ez components separately
- Axial profiles — tighter focus vs low NA
- Quantify the Ez contribution

> **Computation note:** At NA=0.9 the integrands oscillate more rapidly. Each point still uses numerical quadrature. Use 50–80 points; higher resolution increases runtime proportionally.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import sys
sys.path.insert(0, '/Users/raaromero/Projects/Research/optical-diffraction/src')

from optical_diffraction import RichardsWolfSimulator

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 12, 'figure.dpi': 100})

# System parameters — high NA
NA = 0.9
wavelength = 0.532   # microns
n_medium = 1.0
epsilon = 0.0

airy_radius = 0.61 * wavelength / NA
dof = wavelength / NA**2

# Low NA reference for comparison
NA_low = 0.1
airy_radius_low = 0.61 * wavelength / NA_low

print(f"High NA system: NA={NA}, λ={wavelength} μm")
print(f"Paraxial Airy radius: {airy_radius:.3f} μm")
print(f"Paraxial DoF (λ/NA²): {dof:.3f} μm")
print(f"\nRatio to low NA (NA={NA_low}):")
print(f"  Airy radius ratio: {airy_radius_low/airy_radius:.1f}× smaller")
print(f"  DoF ratio: {(wavelength/NA_low**2)/dof:.1f}× shallower")

## 4.1 Focal Plane Radial Intensity — Uniform and Gaussian Inputs

Total intensity I = |Ex|² + |Ey|² + |Ez|² at the focal plane.

In [ ]:
# Radial grid (physical units, microns)
r_max = 4.0 * airy_radius
r = np.linspace(0, r_max, 70)
z_focal = np.zeros_like(r)

configs = [
    ('uniform', 0.0, 'Uniform', 'tab:blue', '-'),
    ('gaussian', 1.0, 'Gaussian α=1', 'tab:orange', '-'),
    ('gaussian', 2.0, 'Gaussian α=2', 'tab:green', '-'),
    ('gaussian', 4.0, 'Gaussian α=4', 'tab:red', '-'),
]

fig, ax = plt.subplots(figsize=(9, 5))

intensities = {}
for (field_type, trunc, label, color, ls) in configs:
    sim = RichardsWolfSimulator(
        wavelength=wavelength,
        numerical_aperture=NA,
        n_medium=n_medium,
        polarization='x',
        input_field=field_type,
        truncation_coeff=trunc,
    )
    Ex, Ey, Ez = sim.compute_field(r, z_focal)
    I = np.abs(Ex)**2 + np.abs(Ey)**2 + np.abs(Ez)**2
    I_norm = I / I.max()
    intensities[label] = (I_norm, Ex, Ey, Ez, I.max())
    ax.plot(r / airy_radius, I_norm, color=color, ls=ls, lw=2, label=label)

ax.axvline(x=1.0, color='black', ls=':', lw=1.2, alpha=0.6, label='Paraxial Airy radius')
ax.set_xlabel('r / r$_{Airy}^{paraxial}$')
ax.set_ylabel('Normalized total intensity I/I$_{max}$')
ax.set_title(f'Focal Plane Radial Intensity — High NA (NA={NA}), Circular Aperture')
ax.set_xlim(0, r_max / airy_radius)
ax.set_ylim(0, 1.05)
ax.legend()
plt.tight_layout()
plt.show()

print("Note: At high NA, the actual focal spot is tighter than the paraxial Airy prediction.")

## 4.2 Field Components Ex, Ey, Ez — Vectorial Effects at High NA

For x-polarized input at NA=0.9, the vectorial effects are significant:
- **Ex**: main component, dominates near the optical axis
- **Ey**: cross-polarized component, zero on axis but non-zero off-axis
- **Ez**: longitudinal (axial) component, can be ~10–20% of Ex at NA=0.9

In [ ]:
# Use uniform illumination to show clearest vectorial structure
sim_unif = RichardsWolfSimulator(
    wavelength=wavelength, numerical_aperture=NA, n_medium=n_medium,
    polarization='x', input_field='uniform', truncation_coeff=0.0,
)
Ex_u, Ey_u, Ez_u = sim_unif.compute_field(r, z_focal)

Ix_u = np.abs(Ex_u)**2
Iy_u = np.abs(Ey_u)**2
Iz_u = np.abs(Ez_u)**2
I_tot_u = Ix_u + Iy_u + Iz_u
scale_u = I_tot_u.max()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Individual components normalized to total peak
component_data = [
    (Ix_u, '|Ex|²', 'tab:blue'),
    (Iy_u, '|Ey|²', 'tab:orange'),
    (Iz_u, '|Ez|²', 'tab:red'),
]

for ax, (I_comp, comp_label, color) in zip(axes, component_data):
    ax.plot(r / airy_radius, I_comp / scale_u, color=color, lw=2.5, label=comp_label)
    ax.plot(r / airy_radius, I_tot_u / scale_u, color='black', lw=1.5, ls='--',
            alpha=0.5, label='Total')
    ax.axvline(x=1.0, color='gray', ls=':', lw=1.0)
    ax.set_xlabel('r / r$_{Airy}$')
    ax.set_ylabel('Intensity / I$_{max}^{total}$')
    ax.set_title(f'{comp_label} Component')
    ax.set_xlim(0, r_max / airy_radius)
    ax.legend()

fig.suptitle(f'Field Components at High NA (NA={NA}) — x-Polarized Uniform Input', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print("Peak intensity fractions:")
print(f"  |Ex|²_max / I_total = {Ix_u.max()/scale_u:.4f}")
print(f"  |Ey|²_max / I_total = {Iy_u.max()/scale_u:.4f}  (cross-polarization at high NA)")
print(f"  |Ez|²_max / I_total = {Iz_u.max()/scale_u:.4f}  (longitudinal, significant at NA={NA})")
print(f"\nFor comparison, paraxial estimate Ez ~ (NA/2)² = {(NA/2)**2:.4f}")

## 4.3 Field Components for Different Input Profiles

In [ ]:
alphas_plot = [1, 2, 4]
colors_alpha = ['tab:orange', 'tab:green', 'tab:red']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Uniform baseline
sim_u = RichardsWolfSimulator(
    wavelength=wavelength, numerical_aperture=NA, n_medium=n_medium,
    polarization='x', input_field='uniform', truncation_coeff=0.0,
)
Ex_u, Ey_u, Ez_u = sim_u.compute_field(r, z_focal)
I_u = np.abs(Ex_u)**2 + np.abs(Ey_u)**2 + np.abs(Ez_u)**2
peak_u = I_u.max()

ax_tot, ax_ex, ax_ey, ax_ez = axes.flatten()

ax_tot.plot(r / airy_radius, I_u / peak_u, color='tab:blue', lw=2, label='Uniform')
ax_ex.plot(r / airy_radius, np.abs(Ex_u)**2 / peak_u, color='tab:blue', lw=2, label='Uniform')
ax_ey.plot(r / airy_radius, np.abs(Ey_u)**2 / peak_u, color='tab:blue', lw=2, label='Uniform')
ax_ez.plot(r / airy_radius, np.abs(Ez_u)**2 / peak_u, color='tab:blue', lw=2, label='Uniform')

for alpha, color in zip(alphas_plot, colors_alpha):
    sim_g = RichardsWolfSimulator(
        wavelength=wavelength, numerical_aperture=NA, n_medium=n_medium,
        polarization='x', input_field='gaussian', truncation_coeff=float(alpha),
    )
    Ex_g, Ey_g, Ez_g = sim_g.compute_field(r, z_focal)
    I_g = np.abs(Ex_g)**2 + np.abs(Ey_g)**2 + np.abs(Ez_g)**2
    peak_g = I_g.max()

    ax_tot.plot(r / airy_radius, I_g / peak_g, color=color, lw=2, label=f'Gaussian α={alpha}')
    ax_ex.plot(r / airy_radius, np.abs(Ex_g)**2 / peak_g, color=color, lw=2, label=f'α={alpha}')
    ax_ey.plot(r / airy_radius, np.abs(Ey_g)**2 / peak_g, color=color, lw=2, label=f'α={alpha}')
    ax_ez.plot(r / airy_radius, np.abs(Ez_g)**2 / peak_g, color=color, lw=2, label=f'α={alpha}')

titles = ['Total Intensity |Ex|²+|Ey|²+|Ez|²', '|Ex|²', '|Ey|² (cross-pol)', '|Ez|² (longitudinal)']
for ax, title in zip([ax_tot, ax_ex, ax_ey, ax_ez], titles):
    ax.axvline(x=1.0, color='black', ls=':', lw=1.0, alpha=0.5)
    ax.set_xlabel('r / r$_{Airy}$')
    ax.set_ylabel('Normalized intensity')
    ax.set_title(title)
    ax.set_xlim(0, r_max / airy_radius)
    ax.legend(fontsize=9)

fig.suptitle(f'High NA (NA={NA}) — Field Components for All Input Profiles', fontsize=13)
plt.tight_layout()
plt.show()

## 4.4 Axial Intensity Profiles — Tight Focus at High NA

In [ ]:
# Axial grid
z_max = 3.0 * dof
z_axial = np.linspace(-z_max, z_max, 60)
r_zero = np.zeros_like(z_axial)

fig, ax = plt.subplots(figsize=(10, 5))

for (field_type, trunc, label, color, ls) in configs:
    sim = RichardsWolfSimulator(
        wavelength=wavelength, numerical_aperture=NA, n_medium=n_medium,
        polarization='x', input_field=field_type, truncation_coeff=trunc,
    )
    Ex, Ey, Ez = sim.compute_field(r_zero, z_axial)
    I = np.abs(Ex)**2 + np.abs(Ey)**2 + np.abs(Ez)**2
    I_norm = I / I.max()
    ax.plot(z_axial / dof, I_norm, color=color, ls=ls, lw=2, label=label)

ax.axvline(x=0, color='black', ls='-', lw=0.8, alpha=0.4)
ax.axhline(y=0.5, color='gray', ls=':', lw=1, alpha=0.6, label='FWHM level')
ax.set_xlabel('z / DoF  (DoF = λ/NA²)')
ax.set_ylabel('Normalized on-axis intensity I(r=0, z)')
ax.set_title(f'Axial Intensity — High NA (NA={NA}), Circular Aperture')
ax.legend()
plt.tight_layout()
plt.show()

print(f"High NA DoF (λ/NA²) = {dof:.3f} μm  vs  Low NA DoF = {wavelength/NA_low**2:.2f} μm")
print(f"DoF reduction factor: {(wavelength/NA_low**2)/dof:.0f}×  (due to NA² in denominator)")

## 4.5 High NA vs Low NA: Focal Spot Comparison

In [ ]:
# Compare high NA and low NA in physical units (microns)
r_compare = np.linspace(0, 6.0 * airy_radius, 70)  # physical microns
z_focal_cmp = np.zeros_like(r_compare)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Uniform illumination comparison
ax = axes[0]
for na_val, ls, label in [(NA, '-', f'NA={NA} (high)'), (NA_low, '--', f'NA={NA_low} (low)')]:
    sim = RichardsWolfSimulator(
        wavelength=wavelength, numerical_aperture=na_val, n_medium=n_medium,
        polarization='x', input_field='uniform', truncation_coeff=0.0,
    )
    r_use = np.linspace(0, 6.0 * (0.61 * wavelength / na_val), 70)
    z_use = np.zeros_like(r_use)
    Ex, Ey, Ez = sim.compute_field(r_use, z_use)
    I = np.abs(Ex)**2 + np.abs(Ey)**2 + np.abs(Ez)**2
    I_norm = I / I.max()
    ax.plot(r_use, I_norm, ls=ls, lw=2.5, label=label)

ax.set_xlabel('Radial position r (μm)')
ax.set_ylabel('Normalized intensity')
ax.set_title('Focal Spot Size: High NA vs Low NA (Uniform)')
ax.legend()

# Gaussian α=2 comparison
ax = axes[1]
for na_val, ls, label in [(NA, '-', f'NA={NA} (high)'), (NA_low, '--', f'NA={NA_low} (low)')]:
    sim = RichardsWolfSimulator(
        wavelength=wavelength, numerical_aperture=na_val, n_medium=n_medium,
        polarization='x', input_field='gaussian', truncation_coeff=2.0,
    )
    r_use = np.linspace(0, 6.0 * (0.61 * wavelength / na_val), 70)
    z_use = np.zeros_like(r_use)
    Ex, Ey, Ez = sim.compute_field(r_use, z_use)
    I = np.abs(Ex)**2 + np.abs(Ey)**2 + np.abs(Ez)**2
    I_norm = I / I.max()
    ax.plot(r_use, I_norm, ls=ls, lw=2.5, label=label)

ax.set_xlabel('Radial position r (μm)')
ax.set_ylabel('Normalized intensity')
ax.set_title('Focal Spot Size: High NA vs Low NA (Gaussian α=2)')
ax.legend()

fig.suptitle('High NA vs Low NA — Physical Focal Spot Size (λ=532 nm)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print(f"Airy radius at NA={NA}: {airy_radius:.3f} μm")
print(f"Airy radius at NA={NA_low}: {airy_radius_low:.3f} μm")
print(f"Spot size reduction: {airy_radius_low/airy_radius:.0f}×")

## Summary

| Observable | NA=0.1 (low) | NA=0.9 (high) | Effect |
|---|---|---|---|
| Airy radius | ~3.24 μm | ~0.36 μm | 9× smaller |
| DoF (λ/NA²) | ~53 μm | ~0.66 μm | 81× shallower |
| Ez/Ex peak ratio | ~0.25% | ~10–20% | Strong longitudinal field |
| Ey (cross-pol) | Negligible | Non-negligible | Depolarization at focus |
| Scalar theory valid? | Yes | No — vectorial required | |

At NA=0.9 the Richards-Wolf vectorial treatment is essential. The focal spot is significantly tighter than the paraxial Airy prediction and the longitudinal Ez component contributes a notable fraction of the total intensity. The next notebook adds annular apertures at high NA.